# 03 — Run full-universe TF15 on Kaggle T4

This local launcher submits the Kaggle runner, monitors its status, and
downloads the resulting CSV/parquet files automatically. Kaggle OAuth is
required only once on this computer; later runs do not require opening Kaggle.

## One-time login

Run the next cell only when this computer has not authenticated with Kaggle.
It opens Kaggle authorization once. Do not paste or commit a token here.

In [ ]:
from pathlib import Path
import subprocess

HERE = Path.cwd().resolve()
if HERE.name != "Daily Screener":
    HERE = HERE / "Daily Screener"
assert (HERE / "launch_kaggle_tf15.py").exists(), f"Open from the ISTL repository: {HERE}"

# Uncomment for the first login only, then follow the authorization prompt:
# subprocess.run([str(HERE / ".venv/bin/kaggle"), "auth", "login"], check=True)

## Submit, monitor, and download

In [ ]:
import importlib.util

module_path = HERE / "launch_kaggle_tf15.py"
spec = importlib.util.spec_from_file_location("kaggle_launcher", module_path)
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)

KAGGLE_USERNAME = launcher.configured_username()
POLL_SECONDS = 30
if not KAGGLE_USERNAME:
    raise ValueError("Isi KAGGLE_USERNAME di file .env.kaggle.local")

downloaded_to = launcher.submit_and_wait(KAGGLE_USERNAME, poll_seconds=POLL_SECONDS)
downloaded_to

In [ ]:
import pandas as pd
from IPython.display import display

ranking_files = list(downloaded_to.rglob("ranking_all_tf15_first_bar_*.csv"))
if not ranking_files:
    raise FileNotFoundError(f"Ranking CSV tidak ditemukan di {downloaded_to}")
kaggle_ranking = pd.read_csv(ranking_files[0])
display(kaggle_ranking.head(30))